In [1]:
!jupyter-nbextension enable nglview --py --sys-prefix

zsh:1: command not found: jupyter-nbextension


In [9]:
import MDAnalysis as mda
import nglview as nv
import numpy as np
import pandas as pd


## Read a specified number of molecules from `npt-HK4.gro`

In [10]:
def extract_molecules_gro(input_file, output_file, num_molecules=10):
    '''This function extracts a specified number of unique molecules from a .gro file.'''

    with open(input_file, 'r') as f:
        lines = f.readlines()
    
    # First two lines: title and atom count
    header = lines[:2]
    atom_lines = lines[2:-1]  # skip last line (box vector)
    box_line = lines[-1]
    
    selected_residues = {}
    for line in atom_lines:
        resname = line[:5].strip()  # e.g. "1HK4"
        if resname not in selected_residues:
            if len(selected_residues) >= num_molecules:
                continue
            selected_residues[resname] = []
        selected_residues[resname].append(line)

    selected_atoms = []
    for lines_in_res in selected_residues.values():
        selected_atoms.extend(lines_in_res)

    # Update atom count in header
    header[1] = f"{len(selected_atoms):5d}\n"

    # Write new .gro file
    with open(output_file, 'w') as f:
        f.writelines(header)
        f.writelines(selected_atoms)
        f.write(box_line)  # reuse original box dimensions


In [13]:
extract_molecules_gro("data/npt-HK4.gro", "data/1mol.gro", num_molecules=1)

## Molecules viewer

In [15]:
u = mda.Universe("data/10mol.gro")

# Create viewer
view = nv.show_mdanalysis(u)
view.clear_representations()
view.add_representation("ball+stick")  # or "licorice", "spacefill", etc.
view.center()
view

NGLWidget()

In [8]:
# Some data types in MDAnalysis
print(type(u))

print(type(u.atoms))
print(type(u.atoms[0]))

print(type(u.residues))
print(type(u.residues[0]))

print(type(u.residues[0].atoms))
print(type(u.residues[0].atoms[0]))

<class 'MDAnalysis.core.universe.Universe'>
<class 'MDAnalysis.core.groups.AtomGroup'>
<class 'MDAnalysis.core.groups.Atom'>
<class 'MDAnalysis.core.groups.ResidueGroup'>
<class 'MDAnalysis.core.groups.Residue'>
<class 'MDAnalysis.core.groups.AtomGroup'>
<class 'MDAnalysis.core.groups.Atom'>


## Midpoint of oxygen

My algorithm to determine the midpoint.

- Case 1: If the distance between the positions is less than a threshold (I set it to be 10 since the usual distance is about 4~6), this is the normal case, then we just calculate the midpoint as usual.

- Case 2: If it exceed the treshold, this indicates that it faces the periodic boundary in that directions, we need another formula to determine the correct midpoint.

$$
    \text{midpoint}= 
\begin{cases}
    \frac{x1 + x2}{2},& \text{if } x\le \text{threshold}\\
    \frac{x1 + x2 + L}{2}  \mod{L},              & \text{otherwise}
\end{cases}
$$

The second algorithm was provided by Gemini, which it says it is more standard and physically robust method. I haven't tried to understand yet, but both works. So I'll just stick with the standard version.

### My algorithm

In [9]:
def calculate_midpoint_pbc(p1: np.ndarray, p2: np.ndarray, L: float, threshold: float = 10.0) -> np.ndarray:
    """
    Calculates the midpoint of two 3D position vectors (p1 and p2)
    with periodic boundary conditions, following a specific logical rule.

    This function applies a user-defined threshold to decide which midpoint
    calculation to use for each dimension.

    Args:
        p1 (np.ndarray): A 3-entry NumPy array representing the first particle's position [x, y, z].
        p2 (np.ndarray): A 3-entry NumPy array representing the second particle's position [x, y, z].
        L (float): The length of the cubic simulation box in each dimension.
        threshold (float): The distance threshold. If the absolute difference between
                           p1 and p2 in a dimension is less than this value, a simple
                           average is used. Otherwise, a wraparound formula is applied.

    Returns:
        np.ndarray: A 3-entry NumPy array representing the calculated midpoint.
    """
    # Ensure inputs are numpy arrays for easier vector operations
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)

    # Initialize the midpoint array
    midpoint = np.zeros(3)

    # Loop through each dimension (x, y, z)
    for i in range(3):
        # Calculate the difference for the current dimension
        diff = p2[i] - p1[i]

        # Check the user-defined condition
        if np.abs(diff) < threshold:
            # If difference is small, use simple average
            midpoint[i] = (p1[i] + p2[i]) / 2
        else:
            # If difference is large, use the specified wraparound formula
            # Note: This specific formula may not always yield a physically
            # correct midpoint along the shortest periodic path.
            midpoint[i] = (p1[i] + p2[i] + L) / 2 % L

    return midpoint

### Standard PBC midpoint algorithm

In [10]:
def calculate_midpoint_pbc_standard(p1: np.ndarray, p2: np.ndarray, L: float) -> np.ndarray:
    """
    Calculates the midpoint of two 3D position vectors using the standard
    minimum image convention for periodic boundary conditions.

    This method is generally more robust and physically correct than
    the custom logic above. It finds the shortest distance between the
    particles, considering the periodic boundaries, and then calculates
    the midpoint along that shortest path.

    Args:
        p1 (np.ndarray): A 3-entry NumPy array representing the first particle's position [x, y, z].
        p2 (np.ndarray): A 3-entry NumPy array representing the second particle's position [x, y, z].
        L (float): The length of the cubic simulation box in each dimension.

    Returns:
        np.ndarray: A 3-entry NumPy array representing the calculated midpoint.
    """
    # Ensure inputs are numpy arrays
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)

    # Calculate the naive difference vector
    diff = p2 - p1

    # Apply the minimum image convention to the difference vector.
    # This finds the shortest vector connecting p1 to p2, accounting for boundaries.
    # The `np.round` function rounds to the nearest integer.
    diff_min_image = diff - L * np.round(diff / L)

    # Calculate the midpoint by adding half of the minimum image vector to the first particle
    midpoint = p1 + 0.5 * diff_min_image

    # Wrap the final midpoint position back into the box [0, L)
    midpoint = midpoint % L

    return midpoint

### Examples

In [11]:
# --- Example Usage ---

# Define the box length
L = 100.0

# Define a threshold for the first function
threshold = 10.0

# Example 1: Particles are close together (diff < threshold)
pos1_close = np.array([5.0, 50.0, 20.0])
pos2_close = np.array([8.0, 54.0, 25.0])
midpoint_close = calculate_midpoint_pbc(pos1_close, pos2_close, L, threshold)

# Example 2: Particles are far apart (diff > threshold)
pos1_far = np.array([5.0, 50.0, 95.0])
pos2_far = np.array([98.0, 55.0, 2.0])
midpoint_far = calculate_midpoint_pbc(pos1_far, pos2_far, L, threshold)

# --- Standard Method Comparison ---
midpoint_close_standard = calculate_midpoint_pbc_standard(pos1_close, pos2_close, L)
midpoint_far_standard = calculate_midpoint_pbc_standard(pos1_far, pos2_far, L)

print(f"Box length (L): {L}, Threshold: {threshold}\n")

print("--- Example 1: Particles are close ---")
print(f"Position 1: {pos1_close}")
print(f"Position 2: {pos2_close}")
print(f"Midpoint (Custom Logic): {midpoint_close}")
print(f"Midpoint (Standard Method): {midpoint_close_standard}")
print("\n" + "="*50 + "\n")

print("--- Example 2: Particles are far apart ---")
print(f"Position 1: {pos1_far}")
print(f"Position 2: {pos2_far}")
print(f"Midpoint (Custom Logic): {midpoint_far}")
print(f"Midpoint (Standard Method): {midpoint_far_standard}")


Box length (L): 100.0, Threshold: 10.0

--- Example 1: Particles are close ---
Position 1: [ 5. 50. 20.]
Position 2: [ 8. 54. 25.]
Midpoint (Custom Logic): [ 6.5 52.  22.5]
Midpoint (Standard Method): [ 6.5 52.  22.5]


--- Example 2: Particles are far apart ---
Position 1: [ 5. 50. 95.]
Position 2: [98. 55.  2.]
Midpoint (Custom Logic): [ 1.5 52.5 98.5]
Midpoint (Standard Method): [ 1.5 52.5 98.5]


### Apply to 10 oxygen molecules case

In [ ]:
# Create a list to store dictionaries for each oxygen atom.
data_list = []

# Loop through each residue in your ResidueGroup
for residue in u.residues:
    # Find all oxygen atoms in the residue.
    oxygen_atoms = residue.atoms.select_atoms("name O*")

    # Get the positions of the two oxygen atoms
    pos1 = oxygen_atoms[0].position
    pos2 = oxygen_atoms[1].position

    # Calculate the distance between the two oxygen atoms
    distance = np.linalg.norm(pos1 - pos2)

    # Calculate the midpoint using periodic boundary conditions
    midpoint = calculate_midpoint_pbc_standard(pos1, pos2, L=u.dimensions[0])

    # Loop through each oxygen atom found
    for atom in oxygen_atoms:
        # Create a dictionary for this atom's data
        atom_data = {
            'resid': atom.resid,
            'resname': atom.resname,
            'atom_name': atom.name,
            'x': atom.position[0],
            'y': atom.position[1],
            'z': atom.position[2],
            'midpoint': midpoint.tolist(),
            'distance': distance
        }
        data_list.append(atom_data)

# Create the pandas DataFrame from the list of dictionaries
df = pd.DataFrame(data_list)

In [33]:
df_midpoint = df['midpoint'].apply(
    lambda arr: [f"{x:.4f}" for x in arr]
)

# Concatenate along columns
concatenated_df_cols = pd.concat([df[['resid', 'atom_name', 'x', 'y', 'z']], df_midpoint], axis=1)

concatenated_df_cols

,resid,atom_name,x,y,z,midpoint
0,1,O2,17.240000,3.940000,109.709999,"[15.5850, 4.0900, 111.6949]"
1,1,O1,13.930000,4.240000,1.200000,"[15.5850, 4.0900, 111.6949]"
2,2,O2,4.860000,8.230000,106.180000,"[4.4150, 6.5700, 107.8750]"
3,2,O1,3.970000,4.910000,109.570000,"[4.4150, 6.5700, 107.8750]"
4,3,O2,9.860000,107.430000,7.820000,"[7.6450, 107.2000, 8.9800]"
5,3,O1,5.430000,106.969994,10.140000,"[7.6450, 107.2000, 8.9800]"
6,4,O2,111.750000,9.540000,2.390000,"[1.5751, 9.7750, 1.3500]"
7,4,O1,3.880000,10.010000,0.310000,"[1.5751, 9.7750, 1.3500]"
8,5,O2,55.590000,3.260000,59.250000,"[55.4400, 1.1051, 57.7600]"
9,5,O1,55.289997,111.429993,56.269997,"[55.4400, 1.1051, 57.7600]"
